# Tutorial MHW
## Aprendiendo a usar la libreria mhw (marine heat waves) en Python  

Tutorial para el uso de la libreria/scripts para el calculo y analisis de olas de calor/frio marino

1. Librerias necesarias y formas de importar mhw
2. Calculo de MHW con la libreria
3. Ejemplos de calculo de estadisticos 

## Librerias necesarias y usando MHW

In [ ]:
import sys
import os

import numpy as np
import xarray as xr
import pandas as pd

try:
    """
    Si la libreria esta instalada
    """
    import mhwpy.datasets_utils as dsu # control de dataset facilitando el uso, no obligatorio
    import mhwpy.mhw_core as mhw # core del script 
    import mhwpy.time_utils as timeu # herramientas y utilidades del manejo del tiempo
    
except ImportError:
    """
    Si la libreria no se instalado o no desea ser instalada. Importante,
    esto es posible si y solo si la notebook esta ubicada en ./mwh/notebooks/
    """
    # Ruta ubicacion scripts mhw
    ruta_mhw = os.path.abspath(os.path.join('..', 'mhw'))

    # Agregamos la ruta al sistema
    if ruta_mhw not in sys.path:
        sys.path.append(ruta_mhw)
    
    import datasets_utils as dsu
    import mhw_core as mhw
    import time_utils as timeu


import warnings
# Ignora warnings especificamente de numpy y la existencia de arrays con nans
warnings.filterwarnings("ignore", message="All-NaN slice encountered")

%load_ext autoreload
%autoreload 2

In [ ]:
# Nombres de archivos que usaran, se recomienda definirlos con anterioridad

## nf_sst: nombre de la fuente de SST (.nc), ubicarla en la carpeta ./mhw/SST/
nf_sst = "ncdcOisst21AggLat-55-335Lon-70-495_Till31dic24.nc"

### Salvo que se indique lo contrario, los datasets calculados se guardaran en: ./mhw/notebooks/datasets 
## Nombre con el cual se guardara la climatologia calculada 
nf_clim = "pcaso_clim.nc"
## Nombre con el cual se guarda el array  de olas de calor como dataset
nf_waves = "pcaso_waves.nc"
## Nombre con el cual se guarda el .csv stock de las olas de calor
nf_df_waves = "pcaso_df_waves.csv"

## Calculo de MHW

Para el calculo de Marine Heat Waves se realizan 3 pasos: 
1. Calculo de climatologia, anomalias, sst del percentil 90 y se construye una mascara de posibles olas de calor/frio (es decir, aquellos puntos donde la temperatura es superior al percentil 90).
2. A partir de la mascara de posibles olas de calor/frio se definen aquellos eventos que si son olas de calor a partir de lo enunciado por por Hobday et al. (2016) (con algunas libertades de configuracion).
3. Se crea un DataFrame como "stock" de olas de calor (guarndo lat, lon, tiempo inicial, tiempo final, duracion y metricas de las anomalias)

Durante todo el proceso, cada paso genera un archivo que puede ser explorado para analisis o exploraciones particulares sobre el fenomeno estudiado. 

### Paso 1: Construyendo climatologia y las posibles MHW

In [ ]:
# Elegimos los limites aplicados al dataset lon1, lon2, lat1, lat2
region_bands = [290.125, 310.625, -54.875, -33.375] 
date_init = "1982-01-01T00:00:00"
date_end = "2024-12-31T23:59:59"

In [ ]:
"""
Calculamos inicilamente las "posibles" MHW, 
para ello calcula climatolgia, SST del percentil 90, anomalias de SST 
y, de ahi, una mascara de posibles MHW. 
Para esto utilizamos la funcion: get_climatology_and_anomialies,
cuyos parametros son: 

    sst_da : xr.DataArray
        DataArray de SST con dimensión temporal 'time' (y típicamente 'lat', 'lon').
    clim_year_init : int, optional
        Año de inicio para el período de climatología base. Por defecto 1991.
    clim_year_end : int, optional
        Año de fin para el período de climatología base. Por defecto 2020.
    window_time : int, optional
        Ancho de la ventana temporal en días (hacia adelante y hacia atrás). 
        Por defecto 5 (crea una ventana total de 11 días).
    percentil: float, optional
        Percentil a calcularse. Por defecto es 0.9,
    coldwaves: boolean, optional
        Tipo de ola a calcular. Por defecto, olas de calor
"""

# Si ya utilice el script, chequea que tenga un dataset existente con ese nombre 
if dsu.exist_file(dsu.join_path(dsu.DIR_DATASETS, nf_clim)):
    ds_clim = xr.open_dataset(dsu.join_path(dsu.DIR_DATASETS, nf_clim))
else:
    """
    Lo primero que hacemos es cargar el Dataset de SST,
    esta configurado para buscarlo en la carpeta ./SST/ 

    en el M. Luz Clara et al el producto SST utilizado fue:


    por lo tanto el sistema esta pensando para ese producto, aunque funciona
    correctamente para todo producto siempre y cuando se tenga cuidado el nombre
    de las dimensiones time/lat/lon
    """
    # Carga datos de SST de ../SST/{nf_sst}
    ds = xr.open_dataset(dsu.join_path(dsu.DIR_SST, nf_sst))
    
    """ 
    Importante: Chequear unidades de longitud y nombre de variables! 
    (lat, lon y time) dependera del SST elegido! 
    """
    # Limito los datos por LAT/LON 
    cond_lat = (ds.latitude > region_bands[2]) & (ds.latitude < region_bands[3])
    cond_lon = (ds.longitude > region_bands[0]) & (ds.longitude < region_bands[1])
    ds = ds.where(cond_lon & cond_lat, drop=True)
    
    # Limito los datos por time
    cond_time_inf = ds.time >= np.datetime64(date_init) 
    cond_time_sup = ds.time <= np.datetime64(date_end)
    ds = ds.where((cond_time_inf & cond_time_sup), drop=True)
    
    # Calculo climatologia y anomalias con get_climatology_and_anomalies
    # el proceso puede durar 
    ds_clim = mhw.get_climatology_and_anomalies(ds.sst)
    dsu.save_dataset_as_nc(ds_clim, name_file=nf_clim)
    ds.close()

### Paso 2: Calculando las MHW 

In [ ]:
"""
Una vez que tenemos "las posibles marine hate waves" chequeamos cuales
de estos eventos anomalos son olas de calor, para ello se utiliza el criterio de
de Hobday et al. (2016). 

Parámetros
    ----------
    posible_mhw_mask: : xr.DataArray
        Array booleano (o numérico) 3D (time, lat, lon). True/1 si supera un dado percentil.
    min_size_mhw : int, opcional
        Duración mínima del evento en días. Por defecto 5.
    max_gap : int, opcional
        Brecha máxima de días consecutivos permitida por debajo del p90
        para unir dos eventos contiguos. Por defecto 2.

"""
# si ya usamos el script con anterior, usa ese archivo
if dsu.exist_file(dsu.join_path(dsu.DIR_DATASETS, nf_waves)):
    waves = xr.open_dataset(dsu.join_path(dsu.DIR_DATASETS, nf_waves))
else:
    waves = mhw.get_marine_heat_waves(ds_clim.posible_mhw_mask)
    dsu.save_dataset_as_nc(waves, name_file=nf_waves)

### Paso 3: Armando stock de MHW y estadisticos asociados

In [ ]:
"""
Una vez que tenemos el DataArray de olas de calor, creamos un stock de olas de
calor como DataFrame de pandas. Este va a guardar siempre:

latitud, longitud, tiempo inicio, tiempo final, duracion ola 

si se le da agrega el dataset de anomalias, le agregamos: 
maximo de anomalia, minimo de anomalia, anomalia media, variacion de la anomalia,
anomalia acumulada, tiempo de crecimiento de la mhw (dias que tarde en llegar al maximo), 
tiempo de decaimiento de la mhw (dias que tardo en terminar el evento una vez llegado al pico 
maximo de anomalia)

si se calcularon mcw agregar: coldwaves=True
"""

# si ya usamos el script con anterior, usa ese archivo
if dsu.exist_file(dsu.join_path(dsu.DIR_DATASETS, nf_df_waves)):
    df_waves = pd.read_csv(dsu.join_path(dsu.DIR_DATASETS, nf_df_waves))
else:
    if type(waves)==xr.Dataset:
        df_waves = mhw.waves_to_dataframe(waves.waves, anomalies=ds_clim.anomalies)
    elif type(waves)==xr.DataArray:
        df_waves = mhw.waves_to_dataframe(waves, anomalies=ds_clim.anomalies)
    else:
        raise "ups! problemas con el archivo waves"
    dsu.save_dataframe_as_csv(df_waves, name_file=nf_df_waves)

## Estadisticos para analisis de tendencias y/o comportamiento

La estregia para todo los estadisticos que se calculen sera similar: agrupamos por periodo de tiempo elegido,
latitud y longitud. 

Notar que la misma estrategia se puede utilizar para estimar valores regionales, es decir,  agrupar por conjuntos de lat/lon.

### Ej 1: Anuales

In [ ]:
# from time_utils as timeu libreria para faciltar la manipulacion de dates conociendo los tipos posibles
# del scritp y su carga
df_waves["year"] = df_waves.t0.apply(timeu.extract_year)

In [ ]:
# Agrupación y aplanado de columnas 
gv = df_waves.groupby(["year", "lat", "lon"]).agg(
    {
        "duration": ["mean", "count"],
        "max_anom": ["max", "mean", "min"],
        "min_anom": ["mean", "max", "min"],
        "acc_anom": ["mean", "max", "std", "sum"],
        "growth_anom": ["mean", "max"],
        "decline_anom": ["mean", "min", "max"]
    }
)
#. Renombrar columnas dataframe agrupado
gv.columns = [f"{c1}_{c2}" for c1, c2 in gv.columns]

#  Transformo mi dataframe agrupado en un dataset 
ds_anual_stats = gv.to_xarray()

In [ ]:
# Renombramo columnas dataset
ds_anual_stats = ds_anual_stats.rename({
    "duration_count": "total_events",
    "duration_mean": "duration",
    "max_anom_max": "anom_max",
    "max_anom_mean": "mean_of_max_anomalies",
    "min_anom_mean":  "mean_of_min_anomalies",
    "acc_anom_mean": "mean_of_acc_anomalies",
    "acc_anom_max": "max_of_acc_anomalies",
    "acc_anom_std": "variation_of_acc_anomalies",
    "acc_anom_sum": "total_acc_anomalies"}
)

In [ ]:
# lo guardamos! 
dsu.save_dataset_as_nc(ds_anual_stats, name_file="pcaso_anual_stats.nc")

### Ej 2: Estacionales

In [ ]:
df_waves["year"] = df_waves.t0.apply(timeu.extract_year)
df_waves["season"] = df_waves.t0.apply(timeu.extract_season)

In [ ]:
# Agrupación y aplanado de columnas 
gv = df_waves.groupby(["year", "season", "lat", "lon"]).agg(
    {
        "duration": ["mean", "count"],
        "max_anom": ["max", "mean", "min"],
        "min_anom": ["mean", "max", "min"],
        "acc_anom": ["mean", "max", "std", "sum"],
        "growth_anom": ["mean", "max"],
        "decline_anom": ["mean", "min", "max"]
    }
)
#. Renombrar columnas dataframe agrupado
gv.columns = [f"{c1}_{c2}" for c1, c2 in gv.columns]

#  Transformo mi dataframe agrupado en un dataset 
ds_seasonal_stats = gv.to_xarray()

In [ ]:
ds_seasonal_stats = ds_seasonal_stats.rename({
    "duration_count": "total_events",
    "duration_mean": "duration",
    "max_anom_max": "anom_max",
    "max_anom_mean": "mean_of_max_anomalies",
    "min_anom_mean":  "mean_of_min_anomalies",
    "acc_anom_mean": "mean_of_acc_anomalies",
    "acc_anom_max": "max_of_acc_anomalies",
    "acc_anom_std": "variation_of_acc_anomalies",
    "acc_anom_sum": "total_acc_anomalies"}
)

In [ ]:
# lo guardamos! 
dsu.save_dataset_as_nc(ds_seasonal_stats, name_file="pcaso_seasonal_stats.nc")

### Ej 3: Mensuales

In [ ]:
df_waves["year"] = df_waves.t0.apply(timeu.extract_year)
df_waves["month"] = df_waves.t0.apply(timeu.extract_month)

In [ ]:
# Agrupación y aplanado de columnas 
gv = df_waves.groupby(["year", "month", "lat", "lon"]).agg(
    {
        "duration": ["mean", "count"],
        "max_anom": ["max", "mean", "min"],
        "min_anom": ["mean", "max", "min"],
        "acc_anom": ["mean", "max", "std", "sum"],
        "growth_anom": ["mean", "max"],
        "decline_anom": ["mean", "min", "max"]
    }
)
#. Renombrar columnas dataframe agrupado
gv.columns = [f"{c1}_{c2}" for c1, c2 in gv.columns]

#  Transformo mi dataframe agrupado en un dataset 
ds_monthly_stats = gv.to_xarray()

In [ ]:
ds_monthly_stats = ds_monthly_stats.rename({
    "duration_count": "total_events",
    "duration_mean": "duration",
    "max_anom_max": "anom_max",
    "max_anom_mean": "mean_of_max_anomalies",
    "min_anom_mean":  "mean_of_min_anomalies",
    "acc_anom_mean": "mean_of_acc_anomalies",
    "acc_anom_max": "max_of_acc_anomalies",
    "acc_anom_std": "variation_of_acc_anomalies",
    "acc_anom_sum": "total_acc_anomalies"}
)

In [ ]:
# lo guardamos! 
dsu.save_dataset_as_nc(ds_monthly_stats, name_file="pcaso_monthly_stats.nc")